## Importar bibliotecas

In [ ]:
import colorsys
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib as mpl
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import pytz
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose, STL

## Funções

In [ ]:
def deslocar_utc_para_fuso_fixo(df, tz_str, coluna_datetime=None, reference_date='2025-01-01', keep_tz=False):

    timezone = pytz.timezone(tz_str)

    reference_timestamp = pd.to_datetime(reference_date)
    if reference_timestamp.tzinfo is None:
        reference_timestamp = reference_timestamp.tz_localize('UTC')
    else:
        reference_timestamp = reference_timestamp.tz_convert('UTC')

    fixed_offset = reference_timestamp.astimezone(timezone).utcoffset()
    if fixed_offset is None:
        raise ValueError(f"Não foi possível determinar offset UTC para {tz_str} em {reference_date!r}.")
    offset_timedelta = pd.Timedelta(fixed_offset)

    df = df.copy()

    def aplicar_deslocamento(data_serie):
        # Converte para datetime UTC naïve
        datetime_utc_naive = pd.to_datetime(data_serie, utc=True).dt.tz_localize(None)
        datetime_shifted = datetime_utc_naive + offset_timedelta
        if keep_tz:
            return datetime_shifted.dt.tz_localize(timezone)
        else:
            return datetime_shifted

    if coluna_datetime is None:
        index_datetime_utc_naive = pd.to_datetime(df.index, utc=True).tz_localize(None)
        index_shifted = index_datetime_utc_naive + offset_timedelta
        df.index = index_shifted.tz_localize(timezone) if keep_tz else index_shifted
    else:
        df[coluna_datetime] = aplicar_deslocamento(df[coluna_datetime])

    return df

def prepare_time_features(data):
    dt_index = pd.DatetimeIndex(data.index)
    if isinstance(data, pd.Series):
        df = data.to_frame(name=data.name)
    else:
        df = data.copy()
    
    df['ANO'] = dt_index.year
    df['MES'] = dt_index.month
    map_meses = {1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr', 5: 'Mai', 6: 'Jun', 7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'}
    df['MES_str'] = df['MES'].map(map_meses)
    df['DIA_str'] = dt_index.strftime('%a')
    map_dias = {'Mon': 'Seg', 'Tue': 'Ter', 'Wed': 'Qua', 'Thu': 'Qui', 'Fri': 'Sex', 'Sat': 'Sab', 'Sun': 'Dom'}
    df['DIA_str'] = df['DIA_str'].map(map_dias)
    df['SEMANA'] = dt_index.isocalendar().week
    df['HORA'] = dt_index.hour
    df['DIA'] = dt_index.dayofweek
    df['ANO_MES'] = dt_index.strftime('%Y_%m')
    
    return df

def generate_distinct_colors(n, hue_start=0.0, hue_end=0.85):
    colors = []
    for i in range(n):
        hue = hue_start + (hue_end - hue_start) * i / max(n - 1, 1)
        saturation = 0.9
        value = 0.9
        rgb = colorsys.hsv_to_rgb(hue, saturation, value)
        hex_color = '#%02x%02x%02x' % (int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
        colors.append(hex_color)
    return colors

def plot_annual(series):
    df = prepare_time_features(series)
    col_name = series.name

    df_annual = df[['MES', 'ANO', col_name]].dropna().groupby(['MES', 'ANO']).mean().reset_index()
    anos = sorted(df_annual['ANO'].unique())
    color_map_anos = generate_distinct_colors(len(anos))

    fig_annual = go.Figure()
    for i, ano in enumerate(anos):
        df_ano = df_annual[df_annual['ANO'] == ano]
        fig_annual.add_trace(go.Scatter(
            x=df_ano['MES'],
            y=df_ano[col_name],
            mode='lines+markers',
            name=str(ano),
            line=dict(color=color_map_anos[i], width=3)
        ))
    fig_annual.update_layout(
        title=f"Gráfico Sazonal - Consumo Mensal - {col_name}",
        xaxis=dict(
            title='Mês',
            tickmode='array',
            tickvals=list(range(1, 13)),
            ticktext=['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
                      'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
        ),
        yaxis=dict(title='Consumo [MW]'),
        legend_title_text='Ano',
        height=700, width=1200
    )
    return fig_annual

def plot_weekly(series):
    df = prepare_time_features(series)
    col_name = series.name

    df_weekly = df[['MES_str', 'DIA_str', col_name, 'DIA']].dropna() \
                  .groupby(['DIA_str', 'MES_str', 'DIA']).mean().reset_index()
    df_weekly = df_weekly.sort_values(by='DIA')
    meses = [mes for mes in ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
             if mes in df_weekly['MES_str'].values]
    color_map_meses = generate_distinct_colors(len(meses))

    order_days = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab', 'Dom']
    df_weekly['DIA_str'] = pd.Categorical(df_weekly['DIA_str'], categories=order_days, ordered=True)
    df_weekly = df_weekly.sort_values(by='DIA_str')

    fig_weekly = go.Figure()
    for i, mes in enumerate(meses):
        df_mes = df_weekly[df_weekly['MES_str'] == mes]
        fig_weekly.add_trace(go.Scatter(
            x=df_mes['DIA_str'],
            y=df_mes[col_name],
            mode='lines+markers',
            name=mes,
            line=dict(color=color_map_meses[i], width=3)
        ))
    fig_weekly.update_layout(
        title=f"Gráfico Sazonal - Consumo Semanal por Mês - {col_name}",
        xaxis=dict(
            title='Dia da Semana',
            categoryorder='array',
            categoryarray=order_days
        ),
        yaxis_title='Consumo [MW]',
        legend_title_text='Mês',
        height=700, width=1200
    )
    return fig_weekly

def plot_daily(series):
    df = prepare_time_features(series)
    col_name = series.name

    df_daily = df[['HORA', 'DIA_str', col_name]].dropna().groupby(['HORA', 'DIA_str']).mean().reset_index()
    order_days = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab', 'Dom']
    df_daily['DIA_str'] = pd.Categorical(df_daily['DIA_str'], categories=order_days, ordered=True)
    df_daily = df_daily.sort_values(by=['DIA_str', 'HORA'])

    dias_unicos = list(df_daily['DIA_str'].cat.categories)
    color_map_dias = generate_distinct_colors(len(dias_unicos))

    fig_daily = go.Figure()
    for i, dia in enumerate(dias_unicos):
        df_dia = df_daily[df_daily['DIA_str'] == dia]
        fig_daily.add_trace(go.Scatter(
            x=df_dia['HORA'],
            y=df_dia[col_name],
            mode='lines+markers',
            name=dia,
            line=dict(color=color_map_dias[i], width=3)
        ))
    fig_daily.update_layout(
        title=f"Gráfico Sazonal - Consumo Diário por Hora - {col_name}",
        xaxis=dict(title='Hora', dtick=1),
        yaxis=dict(title='Consumo [MW]'),
        legend_title_text='Dia da Semana',
        height=700, width=1200
    )
    return fig_daily

def box_plot_annual(series):
    df = prepare_time_features(series)
    col_name = series.name
    df_annual = df[['MES', 'ANO', 'MES_str', col_name]].dropna()
    df_annual['MES_str'] = pd.Categorical(df_annual['MES_str'],
                                         categories=['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
                                                     'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'],
                                         ordered=True)
    df_annual = df_annual.sort_values(by='MES')

    color_palette = generate_distinct_colors(len(df_annual['MES_str'].cat.categories))

    fig_box_annual = go.Figure()
    for i, mes in enumerate(df_annual['MES_str'].cat.categories):
        df_mes = df_annual[df_annual['MES_str'] == mes]
        fig_box_annual.add_trace(go.Box(
            y=df_mes[col_name],
            name=mes,
            boxmean='sd',
            marker_color=color_palette[i]
        ))
    fig_box_annual.update_layout(
        title=f"Box Plot Anual - Consumo Mensal - {col_name}",
        xaxis_title='Mês',
        yaxis_title='Consumo [MW]',
        height=700, width=1200
    )
    return fig_box_annual

def box_plot_weekly(series):
    df = prepare_time_features(series)
    col_name = series.name

    df_weekly = df[['DIA_str', 'DIA', col_name]].dropna()
    order_days = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab', 'Dom']
    df_weekly['DIA_str'] = pd.Categorical(df_weekly['DIA_str'], categories=order_days, ordered=True)
    df_weekly = df_weekly.sort_values(by='DIA')

    color_palette = generate_distinct_colors(len(df_weekly['DIA_str'].cat.categories))

    fig_box_weekly = go.Figure()
    for i, dia in enumerate(df_weekly['DIA_str'].cat.categories):
        df_dia = df_weekly[df_weekly['DIA_str'] == dia]
        fig_box_weekly.add_trace(go.Box(
            y=df_dia[col_name],
            name=dia,
            boxmean='sd',
            marker_color=color_palette[i]
        ))
    fig_box_weekly.update_layout(
        title=f"Box Plot Semanal - Consumo por Dia da Semana - {col_name}",
        xaxis_title='Dia da Semana',
        yaxis_title='Consumo [MW]',
        height=700, width=1200
    )
    return fig_box_weekly

def box_plot_daily(series):
    df = prepare_time_features(series)
    col_name = series.name

    df_daily = df[['HORA', col_name]].dropna()
    horas = sorted(df_daily['HORA'].unique())

    color_palette = generate_distinct_colors(len(horas))

    fig_box_daily = go.Figure()
    for i, hora in enumerate(horas):
        df_hora = df_daily[df_daily['HORA'] == hora]
        fig_box_daily.add_trace(go.Box(
            y=df_hora[col_name],
            name=str(hora),
            boxmean='sd',
            marker_color=color_palette[i]
        ))
    fig_box_daily.update_layout(
        title=f"Box Plot Diário - Consumo por Hora - {col_name}",
        xaxis_title='Hora',
        yaxis_title='Consumo [MW]',
        height=700, width=1200
    )
    return fig_box_daily

def box_plot_year_month(series):
    df = prepare_time_features(series)
    col_name = series.name

    df_ym = df[['ANO_MES', 'ANO', 'MES', col_name]].dropna()
    df_ym = df_ym.sort_values(by=['ANO', 'MES'])
    anos_meses = df_ym['ANO_MES'].unique()

    color_palette = generate_distinct_colors(len(anos_meses))

    fig_box_ym = go.Figure()
    for i, ym in enumerate(anos_meses):
        df_ym_subset = df_ym[df_ym['ANO_MES'] == ym]
        fig_box_ym.add_trace(go.Box(
            y=df_ym_subset[col_name],
            name=ym,
            boxmean='sd',
            marker_color=color_palette[i]
        ))
    fig_box_ym.update_layout(
        title=f"Box Plot - Consumo por Ano-Mês - {col_name}",
        xaxis_title='Ano-Mês',
        yaxis_title='Consumo [MW]',
        height=700, width=1200
    )
    return fig_box_ym


## Descrição e importação de dados

Os dado aqui utilizados são da ENTSOE (European Network of Transmission System Operators for Electricity) e foram adquiridos por meio do [ENTSOE Transparency Platform](https://transparency.entsoe.eu/). Toda coleta foi feita e está descrita no notebook `query_entsoe_data.ipynb`.
Eles consistem em dados horários da França de 2015 a 29/06/2025, sendo eles o Preço Dia Seguinte (Day Ahead Price), a Carga (Load), Geração por Fonte (Generation per Type) e Fluxo na Interconexão (Cross Border Flow). A unidade de medida do preço é EUR/MWh (euros por megawatt-hora) e a unidade de medida da carga, geração e fluxo é MW (megawatts).
De início, será feito o carregamentodos dados, unificação dos dataframes e remoção do efeito do horário de verão para fins de análise.

In [ ]:
# Carregar os dados do arquivo CSV correspondente ao ONS
entsoe_day_ahead_prices = pd.read_csv('../data/day_ahead_prices.csv')
entsoe_flux = pd.read_csv('../data/flux.csv')
entsoe_gen = pd.read_csv('../data/gen.csv')
entsoe_load = pd.read_csv('../data/load.csv')

# Adicionar nas colunas de entsoe_flux e entsoe_gen o prefixo 'Flux_' e 'Gen_', exceto na coluna 'Timestamp'
entsoe_flux = entsoe_flux.add_prefix('Flux_').rename(columns={'Flux_Timestamp': 'Timestamp'})
entsoe_gen = entsoe_gen.add_prefix('Gen_').rename(columns={'Gen_Timestamp': 'Timestamp'})

# Unificar csv's em um único DataFrame
entsoe_data = entsoe_day_ahead_prices.merge(entsoe_load, on='Timestamp', how='inner')
entsoe_data = entsoe_data.merge(entsoe_gen, on='Timestamp', how='inner')
entsoe_data = entsoe_data.merge(entsoe_flux, on='Timestamp', how='inner')

# Exibir as primeiras linhas dos dados originais
print("Dados originais:")
print(entsoe_data.head())

# Converter a coluna 'Timestamp' para o formato datetime e garantir que está em UTC, removendo horário de verão
entsoe_data['Timestamp'] = pd.to_datetime(entsoe_data['Timestamp'], utc=True)

# Definir a coluna 'Timestamp' como índice
entsoe_data.set_index('Timestamp', inplace=True)

# Definir a frequência do índice para garantir que é horário
entsoe_data = entsoe_data.asfreq('h')

# Ajustar o índice para o fuso horário de Paris (UTC+1) sem considerar horário de verão
df = deslocar_utc_para_fuso_fixo(entsoe_data, 'Europe/Paris')

# Exibir as primeiras linhas dos dados ajustados
print("\nDados ajustados (UTC+1, sem horário de verão):")
print(df.head())

## Análise Exploratória de Dados (EDA) e Estatística descritiva

A análise exploratória de dados (EDA) é fundamentas para entender a estrutura, qualidade e características principais de um conjunto de dados antes de avançar para modelagem ou previsão. Aqui são os passos realizados:

- O primeiro passo é descobrir quantos espaços em branco existem, ou seja, quantos dados faltantes temos em cada coluna e no total.

- Em seguida, elimina-se automaticamente aquelas colunas que contêm mais de 20% de falta de informação, pois são elas que podem distorcer qualquer conclusão.

- Logo depois, é conferido se há registros de tempo repetidos: cada instante deve aparecer apenas uma vez para a série manter sua coerência cronológica.

- É avaliado então o menor e o maior horário presente nos dados, e percorrre cronologicamente todo esse intervalo para identificar horários inexistentes e informa quais são os dados faltantes, se houver deverá ser desenvolvida uma ferramenta para preencher.

- Também verificamos se a sequência está ordenada do início ao fim, caso contrário, poderíamos enganar algoritmos que assumem progressão temporal crescente.

- Por fim, calculamos o resumo estatístico básico de cada variável. 
    - Contagem de observações válidas: indica quantos registros estão disponíveis sem valores nulos.
    - Média: aponta o valor ao redor do qual os dados se agrupam, representando a tendência central.
    - Desvio padrão: mede o grau de variação dos valores em relação à média, mostrando a dispersão dos dados.
    - Valor mínimo: identifica o registro de menor valor, auxiliando na detecção de possíveis erros de coleta ou eventos atípicos.
    - Valor máximo: destaca o registro de maior valor, sinalizando picos de demanda ou preços.
    - Quartis (25%, 50% e 75%): dividem a distribuição em quatro partes iguais, permitindo avaliar a assimetria e localizar a faixa onde se concentra a maior parte dos dados.

In [ ]:
# Verificar se existem valores nulos
print("Valores nulos por coluna:")
print(df.isnull().sum())
print(f"Número total de valores nulos: {df.isnull().sum().sum()}")

# Remover colunas com muitos valores nulos, o critério adotado é remover colunas com mais de 20% de valores nulos
limite_nulos = 20 #[%]
colunas_para_remover = df.columns[df.isnull().sum() > ((limite_nulos/100)*len(df))]
df.drop(columns=colunas_para_remover, inplace=True)
if len(colunas_para_remover) > 0:
    print(f"Colunas removidas por excesso de valores nulos (>{limite_nulos}%): {list(colunas_para_remover)}")
    print(f"Número de valores nulos após remoção: {df.isnull().sum().sum()}")

# Verificar se existem timestamps duplicados no índice
print(f"Número de timestamps duplicados: {df.index.duplicated().sum()}")

# Verificar o intervalo de datas
print(f"Intervalo de datas: {df.index.min()} a {df.index.max()}")

# Verificar ausência de datas no intervalo
date_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
missing_dates = date_range.difference(pd.DatetimeIndex(df.index))
print(f"Número de datas ausentes: {len(missing_dates)}")
if len(missing_dates) > 0:
    print("Datas ausentes:")
    print(missing_dates)

# Verificar se está ordenado
if df.index.is_monotonic_increasing:
    print("O índice está ordenado.")
else:
    print("O índice não está ordenado.")

# Verificar estatísticas descritivas
print("Estatísticas descritivas:")
print(df.describe())

### Para a coluna Day Ahead Price, as estatísticas descritivas mostram:

 - Count (91 895): número total de registros de preço, indicando o volume de dados disponíveis para análise.

 - Mean (≈ 77,66 EUR/MWh): preço médio, que reflete o patamar típico de mercado levando em conta tanto preços negativos quanto picos elevados.

 - Std (≈ 90,11 EUR/MWh): elevada variabilidade nos preços, sugerindo grande volatilidade diária influenciada por oferta, demanda e condições de rede.

 - Min (≈ –134,94 EUR/MWh): preço mínimo negativo, indicando momentos em que produtores pagaram para descarregar energia, possivelmente por excesso de geração renovável ou restrições de rede.

 - 25% (≈ 32,12 EUR/MWh): primeiro quartil, mostrando que 25% dos preços foram abaixo desse valor, representando condições de menor demanda ou maior oferta.

 - 50% (≈ 48,00 EUR/MWh): mediana, preço central menos sensível a outliers, que indica que metade dos dias registrou preço inferior a 48 EUR/MWh.

 - 75% (≈ 84,81 EUR/MWh): terceiro quartil, limite acima do qual estão os 25% maiores preços, úteis para avaliar períodos de maior escassez ou picos de demanda.

 - Max (≈ 2 987,78 EUR/MWh): preço máximo extremo, sinalizando eventos atípicos de mercado, como falhas de geração ou picos críticos de demanda.

In [ ]:
# Visualizar todos os dados
fig = go.Figure()
color_data = generate_distinct_colors(len(df.columns))
for i, col in enumerate(df.columns):
    fig.add_trace(go.Scatter(x=df.index,
                             y=df[col],
                             mode='lines',
                             name=col,
                             line=dict(color=color_data[i])))
fig.update_layout(title='Plot de Todos os Dados',
                  xaxis_title='Data',
                  height=700,
                  width=1200)
fig.show()

In [ ]:
df = prepare_time_features(df)

In [ ]:
# Decompor a série temporal para observar tendências e sazonalidades
result = seasonal_decompose(df['Day Ahead Price'], model='additive', period=24)

# Criar subplots empilhados verticalmente, 4 linhas para original + trend + seasonal + resid
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=('Original', 'Tendência', 'Sazonalidade', 'Resíduos'))

# Adicionar as séries para cada componente
fig.add_trace(go.Scatter(x=df['Day Ahead Price'].index, y=df['Day Ahead Price'], mode='lines', name='Original'), row=1, col=1)
fig.add_trace(go.Scatter(x=result.trend.index, y=result.trend, mode='lines', name='Tendência'), row=2, col=1)
fig.add_trace(go.Scatter(x=result.seasonal.index, y=result.seasonal, mode='lines', name='Sazonalidade'), row=3, col=1)
fig.add_trace(go.Scatter(x=result.resid.index, y=result.resid, mode='lines', name='Resíduos'), row=4, col=1)

# Ajustar layout
fig.update_layout(height=800, width=1000, title_text='Decomposição Série Temporal - SUDESTE', showlegend=False)
fig.update_xaxes(title_text='Data/Hora', row=4, col=1)
fig.update_yaxes(title_text='Consumo (MW)', row=1, col=1)
fig.update_yaxes(title_text='Tendência', row=2, col=1)
fig.update_yaxes(title_text='Sazonalidade', row=3, col=1)
fig.update_yaxes(title_text='Resíduos', row=4, col=1)

fig.show()

In [ ]:
stl = STL(df['Day Ahead Price'], period=24)
res = stl.fit()
fig_stl = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=('Original', 'Tendência', 'Sazonalidade', 'Resíduos'))
fig_stl.add_trace(go.Scatter(x=df.index, y=df['Day Ahead Price'], mode='lines', name='Original'), row=1, col=1)
fig_stl.add_trace(go.Scatter(x=res.trend.index, y=res.trend, mode='lines', name='Tendência'), row=2, col=1)
fig_stl.add_trace(go.Scatter(x=res.seasonal.index, y=res.seasonal, mode='lines', name='Sazonalidade'), row=3, col=1)
fig_stl.add_trace(go.Scatter(x=res.resid.index, y=res.resid, mode='lines', name='Resíduos'), row=4, col=1)
fig_stl.update_layout(height=800, width=1000, title_text='Decomposição STL - SUDESTE', showlegend=False)
fig_stl.update_xaxes(title_text='Data/Hora', row=4, col=1)
fig_stl.update_yaxes(title_text='Consumo (MW)', row=1, col=1)
fig_stl.update_yaxes(title_text='Tendência', row=2, col=1)
fig_stl.update_yaxes(title_text='Sazonalidade', row=3, col=1)
fig_stl.update_yaxes(title_text='Resíduos', row=4, col=1)
fig_stl.show()

In [ ]:
# AFC e PACF
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig_acf = plot_acf(df['Day Ahead Price'].dropna(), lags=50, title='Autocorrelação (ACF) - SUDESTE')
fig_acf.show()
fig_pacf = plot_pacf(df['Day Ahead Price'].dropna(), lags=50, title='Autocorrelação Parcial (PACF) - SUDESTE')
fig_pacf.show()

# LAGs
df['SUDESTE_LAG1'] = df['Day Ahead Price'].shift(1)
df['SUDESTE_LAG24'] = df['Day Ahead Price'].shift(24)
df['SUDESTE_LAG168'] = df['Day Ahead Price'].shift(168)  # 7 dias * 24 horas

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

actual = df['Day Ahead Price']
hours = range(0, 24, 4)

for hour in hours:
    plot_pacf(actual[actual.index.hour == hour].diff().dropna(), lags=30, alpha=0.01)
    plt.title(f'PACF - h = {hour}')
    plt.ylabel('Correlation')
    plt.xlabel('Lags')
    plt.show()

In [ ]:
# Seasonal-Trend decomposition using Loess
result = seasonal_decompose(df['Day Ahead Price'], model='additive', period=24)

In [ ]:
def detectar_extremos(df, colunas, limiar_outlier=3):
    """
    Detecta valores zero e outliers em colunas numéricas de um DataFrame.

    Essa função aplica um critério clássico de detecção de outliers baseado na regra empírica
    da distribuição normal (média ± limiar * desvio padrão), onde normalmente limiar=3 significa
    cerca de 99,7% de confiança para valores "normais". Valores fora desse intervalo são considerados outliers.

    Além disso, conta o número de valores exatamente iguais a zero, que podem indicar falhas
    na medição ou dados inválidos dependendo do contexto da série temporal.

    Parâmetros:
    -----------
    df : pandas.DataFrame
        DataFrame contendo as colunas numéricas a serem analisadas.
    colunas : list
        Lista de strings com os nomes das colunas do DataFrame a serem verificadas.
    limiar_outlier : int ou float, padrão 3
        Número de desvios padrão usados para definir os limites de outliers.

    Retorna:
    --------
    dict
        Dicionário onde cada chave é o nome da coluna e o valor é um dicionário com:
            - 'zeros': quantidade de valores iguais a zero na coluna.
            - 'outliers_count': quantidade total de outliers detectados na coluna.
            - 'outliers_indices': lista de índices dos outliers para inspeção.

    Referências:
    ------------
    - Montgomery, D.C., Runger, G.C. Applied Statistics and Probability for Engineers, Wiley, 2010.
    - James, G., Witten, D., Hastie, T., Tibshirani, R. An Introduction to Statistical Learning, Springer, 2013.
    - Han, J., Kamber, M., Pei, J. Data Mining: Concepts and Techniques, Morgan Kaufmann, 2012.

    """
    relatorio = {}
    for col in colunas:
        serie = df[col]
        zero_count = (serie == 0).sum()
        mean = serie.mean()
        std = serie.std()

        # Definir limites para detecção de outliers
        limite_inferior = mean - limiar_outlier * std
        limite_superior = mean + limiar_outlier * std

        # Detectar outliers fora dos limites
        outliers = serie[(serie < limite_inferior) | (serie > limite_superior)]

        relatorio[col] = {
            'zeros': zero_count,
            'outliers_count': outliers.count(),
            'outliers_indices': outliers.index.tolist()
        }
    return relatorio

# Chamar a função detectar_extremos
cols = ['Day Ahead Price']
relatorio = detectar_extremos(df, cols, limiar_outlier=3)

# Exibir um resumo do relatório
for cols, info in relatorio.items():
    print(f"Região: {cols}")
    print(f"  Valores 0 encontrados: {info['zeros']}")
    print(f"  Outliers detectados: {info['outliers_count']}")
    print(f"  Índices dos primeiros outliers: {info['outliers_indices'][:5]}")
    print()


# Análise

## Interpretação Estatística Detalhada dos Valores Descritivos
- Count (223.488): Número total de observações válidas para cada região, confirmando que não há dados ausentes, o que é fundamental para estabilidade estatística e consistência temporal.

- Mean (Média): Indica o consumo médio de energia para cada região no período analisado. No caso do Sudeste (~33.646 MW), reflete a demanda típica, equilibrando consumos baixos e picos. A média é sensível a valores extremos, portanto ajuda a entender o nível geral de consumo.

- Std (Desvio padrão): Mede a dispersão dos dados em relação à média. Um valor alto no Sudeste (≈7.548 MW) revela alta variabilidade do consumo, indicando que existem grande oscilações e picos, possivelmente ligados a fatores externos como clima, atividade econômica e horário. Regiões com menor desvio padrão apresentam consumo mais estável.

- Min (Mínimo): O menor valor registrado, que pode representar:

    - Baixa demanda em horários de mínimo consumo (ex: madrugada).

    - Possíveis registros faltantes ou erros, especialmente se zero ou muito baixo sem padrão esperado.

- 25% (Primeiro quartil): Representa o limite inferior para os 25% menores consumos. Útil para diferenciar períodos de baixa demanda, que podem auxiliar em estudos de eficiência e respostas à redução de carga.

- 50% (Mediana): O valor central que divide os dados ao meio, sendo menos sensível a outliers que a média. Quando próxima da média, como no seu caso, indica distribuição relativamente simétrica do consumo.

- 75% (Terceiro quartil): Limite para os 25% maiores consumos. Valores elevados, como no Sudeste, indicam ocorrência frequente de alta demanda, importante para planejamento de capacidade e picos de energia.

- Max (Máximo): O maior consumo registrado, serve para identificar extremos que podem ser picos de carga por fatores sazonais ou eventos atípicos (ex: ondas de calor, fenômenos econômicos).

## Aplicação Prática
- A diferença entre o 75% e o máximo indica a intensidade e frequência dos picos extremos, essenciais para definir limites de operação segura da rede.

- A mediana próxima à média suporta o uso de modelos que assumem distribuições próximas do normal para estimativas preditivas.

- O desvio padrão elevado no Sudeste implica modelos de previsão precisam capturar alta volatilidade, por exemplo, usando modelagem estocástica ou regressão ajustada a variáveis externas.

# Escrita

## Análise Referenciada dos Dados de Consumo Energético da ONS
Os dados do conjunto representam séries temporais do consumo energético das regiões brasileiras Nordeste, Norte, Sudeste e Sul, ao longo de um grande período com granularidade horária, similar a análises encontradas na literatura acadêmica sobre modelagem e previsão de consumo elétrico.

## Distribuição e Estatísticas Descritivas
O cenário observado, onde a região Sudeste apresenta maior consumo médio (≈33.646 MW) e maior variabilidade (desvio padrão ≈7.548), é condizente com estudos como Silva et al. (2013) e Campos (2008), que mostram que regiões com maior densidade populacional e atividade econômica apresentam consumos médios e variabilidades mais elevadas devido à junção dos setores industrial, comercial e residencial.

A identificação de valores mínimos próximos a zero, especialmente na região Sul, é um ponto que merece investigação, pois pode indicar períodos de baixa demanda ou falhas na medição, como discutido em estudos sobre qualidade de dados no setor energético (Serrano et al., 2022).

## Importância da Análise Temporal
De acordo com Júnior et al. (2018) e Araújo et al. (2013), séries temporais de consumo energético frequentemente exibem componentes sazonais e tendências que devem ser modeladas para previsões confiáveis, especialmente para planejamento do sistema elétrico e gestão da demanda.

A descoberta de picos (percentil 75 significativamente acima da mediana no Sudeste) indica a presença de episódios de alta demanda, que são cruciais para ajustes operacionais e dimensionamento de capacidade, alinhando-se às conclusões de LEON e PESSANHA (2011).

## Recomendações para Modelagem
A robustez e a densidade dos dados suportam o uso de técnicas avançadas de decomposição e modelagem de séries temporais, como métodos Holt-Winters e SARIMA para captura das diversas componentes temporais.

Correção do efeito do horário de verão é essencial para evitar distorções que comprometam a modelagem e casamentos temporais com dados de outras fontes (ex: ENTSOE).

Atenção deverá ser dada à qualidade dos dados, com possível aplicaçãõ de filtros para outliers e validações cruzadas para garantir consistência temporal (Campos, 2008).

## Referências

Silva, J. et al. (2013). Modelagem e previsão de série temporal do consumo de energia elétrica no Nordeste do Brasil. Dissertação de Mestrado, UFPB. [Link](https://dspace.bc.uepb.edu.br/jspui/bitstream/123456789/25769/1/PDF%20-%20Eduardo%20Gomes%20de%20Ara%c3%bajo)

Campos, R.J. (2008). Previsão de séries temporais com aplicações a séries de consumo de energia elétrica. Dissertação de Mestrado, UFMG. [Link](https://repositorio.ufmg.br/server/api/core/bitstreams/019330f5-d68e-4fc6-b371-db707277ffd2/content)

Serrano, A.L.M. et al. (2022). Análise e investigação da capacidade instalada de energia elétrica no Brasil através de modelo paramétrico. Revista Produção Online. [Link](https://www.producaoonline.org.br/rpo/article/view/4549/2162)

Leon, V.E., Pessanha, J. (2011). Decomposição do consumo de energia elétrica residencial nos estados brasileiros. Revista de Energia. [Link](https://sbpe.org.br/index.php/rbe/article/view/708/526)